In [10]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import cell2location

In [11]:
adata_ref = sc.read_h5ad('reference.NBregression.h5ad')

In [12]:
adata_vis = sc.read_h5ad('adata_vis_cellabundance.Rds')

In [13]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:5]

,Bcell,DendriticCells,Endothelial,Epi_normal,Epi_tumor
id,,,,,
ENSG00000243485,0.332799,0.689419,0.475260,0.869007,0.410197
ENSG00000237613,0.243632,0.434621,0.430291,0.586398,0.398066
ENSG00000186092,0.632007,0.475587,0.398321,0.441376,0.226664
ENSG00000238009,0.000345,0.007007,0.003439,0.018365,0.001510
ENSG00000239945,0.347943,0.595761,0.665939,0.646347,0.281798


In [14]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_vis.var_names, inf_aver.index)
adata_vis = adata_vis[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

# prepare anndata for cell2location model
cell2location.models.Cell2location.setup_anndata(adata=adata_vis, batch_key="sample")

In [15]:
# create and train the model
mod = cell2location.models.Cell2location(
    adata_vis, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent
    # hyper-prior which can be estimated from paired histology:
    N_cells_per_location=8,
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
)

In [16]:
mod.view_anndata_setup()

Anndata setup with scvi-tools version 1.3.3.

Setup via `Cell2location.setup_anndata` with arguments:

{
│   'layer': None,
│   'batch_key': 'sample',
│   'labels_key': None,
│   'categorical_covariate_keys': None,
│   'continuous_covariate_keys': None
}

         Summary Statistics         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃     Summary Stat Key     ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│         n_batch          │   4   │
│         n_cells          │ 30207 │
│ n_extra_categorical_covs │   0   │
│ n_extra_continuous_covs  │   0   │
│         n_labels         │   1   │
│          n_vars          │ 18085 │
└──────────────────────────┴───────┘

               Data Registry                
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Registry Key ┃    scvi-tools Location    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      X       │          adata.X          │
│    batch     │ adata.obs['_scvi_batch']  │
│    ind_x     │   adata.obs['_indices']   │
│    labels    │ adata.obs['_scvi_labels'] │
└──────────────┴───────────────────────────┘

                   batch State Registry                   
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃   Source Location   ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['sample'] │  xinpian2  │          0          │
│                     │  xinpian3  │          1          │
│                     │  xinpian4  │          2          │
│                     │  xinpian5  │          3          │
└─────────────────────┴────────────┴─────────────────────┘

                     labels State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['_scvi_labels'] │     0      │          0          │
└───────────────────────────┴────────────┴─────────────────────┘

In [ ]:
mod.train(max_epochs=10000,
          # train using full data (batch_size=None)
          batch_size=None,
          # use all data points in training because
          # we need to estimate cell abundance at all locations
          train_size=1
         )

/disk1/pengweixing/software/miniforge4/envs/scanpy310/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/disk1/pengweixing/software/miniforge4/envs/scanpy310/lib/python3.10/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
/disk1/pengweixing/software/miniforge4/envs/scanpy310/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/disk1/pengweixing/software/miniforge4/envs/scanpy310/lib/python3.10/site-packages/lightning/pytor

Epoch 2/10000:   0%|          | 1/10000 [00:29<80:51:58, 29.11s/it, v_num=1]

/disk1/pengweixing/software/miniforge4/envs/scanpy310/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Epoch 3/10000:   0%|          | 2/10000 [00:56<78:25:30, 28.24s/it, v_num=1, elbo_train=8.85e+8]

/disk1/pengweixing/software/miniforge4/envs/scanpy310/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


...

/disk1/pengweixing/software/miniforge4/envs/scanpy310/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Epoch 8/10000:   0%|          | 7/10000 [03:11<74:50:53, 26.96s/it, v_num=1, elbo_train=8.74e+8]

/disk1/pengweixing/software/miniforge4/envs/scanpy310/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Epoch 9/10000:   0%|          | 8/10000 [03:38<74:52:20, 26.98s/it, v_num=1, elbo_train=8.74e+8]

/disk1/pengweixing/software/miniforge4/envs/scanpy310/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [ ]:
# Save model
mod.save("adata_visum_raw.it10000.model", overwrite=True)

In [ ]:
adata_vis = mod.export_posterior(
    adata_vis, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
)

Sampling local variables, batch:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# plot ELBO loss history during training, removing first 100 epochs from the plot
mod.plot_history(20)
plt.legend(labels=['full data training']);

In [45]:
# add 5% quantile, representing confident cell abundance, 'at least this amount is present',
# to adata.obs with nice names for plotting
adata_vis.obs[adata_vis.uns['mod']['factor_names']] = adata_vis.obsm['q05_cell_abundance_w_sf']

In [46]:
# select one slide
from cell2location.utils import select_slide
slide = select_slide(adata_vis, 'xinpian2')

In [47]:
set(adata_ref.obs['celltype2'])

{'Bcell',
 'DendriticCells',
 'Endothelial',
 'Epi_normal',
 'Epi_tumor',
 'Fibroblasts',
 'Macrophages',
 'Mastcell',
 'Monocytes',
 'Musclecell',
 'NKcell',
 'Neutrophils',
 'Pericyte',
 'Plasma',
 'Tcell'}

In [ ]:
mod = cell2location.models.Cell2location.load('adata_visum_raw.it10000.model', adata_vis)

In [ ]:
convert_legacy_save